# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00


In [3]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("/content/drive/MyDrive/dl_final/")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


## 2. Load and Preprocess Data

In [25]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

In [28]:
splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

overview = []

for name, df in splits.items():
    overview.append({
        "split": name,
        "num_samples": len(df),
        "num_columns": df.shape[1],
        "has_answer": "answer" in df.columns,
        "has_solution": "solution" in df.columns,
    })

overview_df = pd.DataFrame(overview)
display(overview_df)

print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

,split,num_samples,num_columns,has_answer,has_solution
0,train,3109,15,True,True
1,val,1048,15,True,True
2,test,1008,13,False,False


Train columns:
['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']

Test columns:
['id', 'image_path', 'question', 'choices', 'num_choices', 'hint', 'lecture', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


In [29]:
important_cols = ["image_path", "question", "choices", "num_choices", "answer",
                  "hint", "lecture", "solution", "task", "grade", "subject",
                  "topic", "category", "skill"]

missing_rows = []

for name, df in splits.items():
    for col in important_cols:
        if col in df.columns:
            missing_rows.append({
                "split": name,
                "column": col,
                "missing_count": df[col].isna().sum(),
                "missing_ratio": df[col].isna().mean(),
            })

missing_df = pd.DataFrame(missing_rows)
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values(
    ["split", "missing_ratio"], ascending=[True, False]
)

display(missing_df)

,split,column,missing_count,missing_ratio
32,test,hint,214,0.212302
33,test,lecture,183,0.181548
5,train,hint,724,0.232872
7,train,solution,529,0.170151
6,train,lecture,440,0.141525
19,val,hint,232,0.221374
21,val,solution,172,0.164122
20,val,lecture,133,0.126908


In [30]:
def parse_choices(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        return literal_eval(x)
    except Exception:
        return []

for name, df in splits.items():
    df["choices_parsed"] = df["choices"].apply(parse_choices)
    df["choice_len"] = df["choices_parsed"].apply(len)

    mismatch = (df["choice_len"] != df["num_choices"]).sum()
    print(f"{name}: choice_len != num_choices: {mismatch}")

train: choice_len != num_choices: 0
val: choice_len != num_choices: 0
test: choice_len != num_choices: 0


In [31]:
for name, df in {"train": train_df, "val": val_df}.items():
    invalid = (
        (df["answer"].isna()) |
        (df["answer"] < 0) |
        (df["answer"] >= df["num_choices"])
    )
    print(f"{name}: invalid answer count = {invalid.sum()}")

train: invalid answer count = 0
val: invalid answer count = 0


In [33]:
def image_exists(rel_path):
    return (DATA_DIR / rel_path).exists()

for name, df in splits.items():
    missing_img = (~df["image_path"].apply(image_exists)).sum()
    print(f"{name}: missing images = {missing_img}")

train: missing images = 0
val: missing images = 0
test: missing images = 0


In [34]:
def normalized_counts(df, col, topn=None):
    vc = df[col].value_counts(normalize=True)
    if topn is not None:
        vc = vc.head(topn)
    return vc

# num_choices distribution
num_choice_summary = pd.DataFrame({
    name: df["num_choices"].value_counts().sort_index()
    for name, df in splits.items()
}).fillna(0).astype(int)

display(num_choice_summary)

# answer distribution for labeled splits
answer_summary = pd.DataFrame({
    name: df["answer"].value_counts().sort_index()
    for name, df in {"train": train_df, "val": val_df}.items()
}).fillna(0).astype(int)

display(answer_summary)

,train,val,test
num_choices,,,
2,664,244,272
3,1552,508,438
4,783,252,260
5,110,44,38


,train,val
answer,,
0,1124,347
1,1028,372
2,737,247
3,204,75
4,16,7


In [35]:
meta_cols = ["task", "grade", "subject", "topic"]

for col in meta_cols:
    print("=" * 80)
    print(f"{col} distribution, normalized top 10")

    summary = pd.concat(
        {
            "train": normalized_counts(train_df, col, topn=10),
            "val": normalized_counts(val_df, col, topn=10),
        },
        axis=1
    ).fillna(0)

    display(summary)

task distribution, normalized top 10


,train,val
task,,
closed choice,0.987777,0.968511
true-or false,0.012223,0.031489


grade distribution, normalized top 10


,train,val
grade,,
grade8,0.220328,0.220420
grade6,0.196526,0.183206
grade7,0.175298,0.173664
grade4,0.155677,0.164122
grade5,0.113863,0.113550
grade3,0.092956,0.101145
grade2,0.036989,0.034351
grade12,0.006755,0.007634
grade1,0.001608,0.000954


subject distribution, normalized top 10


,train,val
subject,,
natural science,0.728208,0.741412
social science,0.247025,0.231870
language science,0.024767,0.026718


topic distribution, normalized top 10


,train,val
topic,,
biology,0.250563,0.260496
physics,0.196526,0.203244
geography,0.127372,0.123092
science-and-engineering-practices,0.126407,0.108779
chemistry,0.090383,0.086832
earth-science,0.058540,0.077290
us-history,0.055645,0.036260
economics,0.052428,0.061069
writing-strategies,0.018977,0.020038


In [37]:
text_cols = ["question", "hint", "lecture"]

for name, df in splits.items():
    for col in text_cols:
        if col in df.columns:
            df[f"{col}_char_len"] = df[col].fillna("").astype(str).str.len()

    available_text_cols = [col for col in text_cols if col in df.columns]
    df["total_text_char_len"] = df[available_text_cols].fillna("").astype(str).agg(" ".join, axis=1).str.len()

length_rows = []

for name, df in splits.items():
    length_cols = [f"{col}_char_len" for col in text_cols if f"{col}_char_len" in df.columns]
    length_cols.append("total_text_char_len")

    for col in length_cols:
        length_rows.append({
            "split": name,
            "field": col.replace("_char_len", ""),
            "has_text_ratio": (df[col] > 0).mean(),
            "mean": df[col].mean(),
            "median": df[col].median(),
            "p90": df[col].quantile(0.90),
            "max": df[col].max(),
        })

text_length_summary = pd.DataFrame(length_rows)
display(text_length_summary)

,split,field,has_text_ratio,mean,median,p90,max
0,train,question,1.000000,70.700547,62.0,110.0,289
1,train,hint,0.767128,239.220650,153.0,649.0,1845
2,train,lecture,0.858475,664.686073,588.0,1303.0,2003
3,train,total_text,1.000000,976.607269,781.0,2068.4,2846
4,val,question,1.000000,72.378817,62.0,110.0,259
5,val,hint,0.778626,232.136450,153.0,638.3,1785
6,val,lecture,0.873092,665.091603,562.0,1246.0,2003
7,val,total_text,1.000000,971.606870,781.0,2047.1,2747
8,test,question,1.000000,70.989087,62.0,110.0,259
9,test,hint,0.787698,243.547619,153.0,642.3,1910


In [40]:
def inspect_images(df, split_name, cache_dir=DATA_DIR):
    cache_path = cache_dir / f"{split_name}_image_stats.csv"

    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    for rel_path in tqdm(df["image_path"], desc=f"Inspecting {split_name} images"):
        img_path = DATA_DIR / rel_path
        with Image.open(img_path) as img:
            w, h = img.size
            mode = img.mode

        rows.append({
            "image_path": rel_path,
            "width": w,
            "height": h,
            "mode": mode,
        })

    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(cache_path, index=False)
    return stats_df


image_stats = {}

for name, df in splits.items():
    image_stats[name] = inspect_images(df, name)

image_summary_rows = []

for name, stats_df in image_stats.items():
    image_summary_rows.append({
        "split": name,
        "width_mean": stats_df["width"].mean(),
        "width_median": stats_df["width"].median(),
        "width_min": stats_df["width"].min(),
        "width_max": stats_df["width"].max(),
        "height_mean": stats_df["height"].mean(),
        "height_median": stats_df["height"].median(),
        "height_min": stats_df["height"].min(),
        "height_max": stats_df["height"].max(),
        "num_modes": stats_df["mode"].nunique(),
        "modes": ", ".join(stats_df["mode"].value_counts().index.tolist()),
    })

image_summary = pd.DataFrame(image_summary_rows)
display(image_summary)

,split,width_mean,width_median,width_min,width_max,height_mean,height_median,height_min,height_max,num_modes,modes
0,train,445.327758,391.0,68,760,284.646510,239.0,65,595,1,RGB
1,val,446.917939,426.0,66,760,279.509542,239.0,80,595,1,RGB
2,test,438.023810,426.0,93,760,281.848214,237.0,77,595,1,RGB


In [41]:
# Random baseline: randomly choose among valid choices for each validation sample
rng = np.random.default_rng(SEED)
random_preds = [rng.integers(0, n) for n in val_df["num_choices"]]
random_acc = (np.array(random_preds) == val_df["answer"].to_numpy()).mean()

# Majority-answer baseline: always predict the most frequent answer index in train
majority_answer = train_df["answer"].value_counts().idxmax()
majority_preds = np.full(len(val_df), majority_answer)
majority_acc = (majority_preds == val_df["answer"].to_numpy()).mean()

print(f"Random baseline val accuracy: {random_acc:.4f}")
print(f"Majority answer from train: {majority_answer}")
print(f"Majority baseline val accuracy: {majority_acc:.4f}")

Random baseline val accuracy: 0.3378
Majority answer from train: 0
Majority baseline val accuracy: 0.3311


In [43]:
train_small = train_df.sample(n=300, random_state=SEED)
val_small = val_df.sample(n=150, random_state=SEED)

print(f"Saved train_small: {len(train_small)}")
print(f"Saved val_small: {len(val_small)}")

Saved train_small: 300
Saved val_small: 150


In [44]:
eda_summary = {
    "num_samples": {name: len(df) for name, df in splits.items()},
    "num_choices_dist": {
        name: df["num_choices"].value_counts().sort_index().astype(int).to_dict()
        for name, df in splits.items()
    },
    "answer_dist": {
        name: df["answer"].value_counts().sort_index().astype(int).to_dict()
        for name, df in {"train": train_df, "val": val_df}.items()
    },
    "random_baseline_val_acc": float(random_acc),
    "majority_answer": int(majority_answer),
    "majority_baseline_val_acc": float(majority_acc),
    "text_length_summary": text_length_summary.to_dict(orient="records"),
    "image_summary": image_summary.to_dict(orient="records"),
}

summary_path = DATA_DIR / "eda_summary.json"

with open(summary_path, "w") as f:
    json.dump(eda_summary, f, indent=2)

print(f"Saved EDA summary to: {summary_path}")

Saved EDA summary to: /content/drive/MyDrive/dl_final/eda_summary.json


In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Maps have four cardinal directions, or main directions. Those directions are north, south, east, and west.
A compass rose is a set of arrows that point to the cardinal directions. A compass rose usually shows only the first letter of each cardinal direction.
The north arrow points to the North Pole. On most maps, north is at the top of the map.

Question: Which of these states is farthest north?
Choices:
  A. West Virginia
  B. Louisiana
  C. Arizona
  D. Oklahoma
Answer: A


In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=6218, val=2097, test=2017


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/root/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/root/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Prompt:
<image>
Context:
An adaptation is an inherited trait that helps an organism survive or reproduce. Adaptations can include both body parts and behaviors.
The shape of an animal's mouth is one example of an adaptation. Animals' mouths can be adapted in different ways. For example, a large mouth with sharp teeth might help an animal tear through meat. A long, thin mouth might help an animal catch insects that live in holes. Animals that eat similar food often have similar mouths.
Sturgeons eat invertebrates, plants, and small fish. They are bottom feeders. Bottom feeders find their food at the bottom of rivers, lakes, and the ocean.
The 's mouth is located on the underside of its head and points downward. Its mouth is adapted for bottom feeding.
Figure: sturgeon.

Question: Which animal's mouth is also adapted for bottom feeding?
Choices:
  A. discus
  B. armored catfish
Answer:

Model output:






Context:
An adaptation is an inherited trait that helps an organism survive or rep